# NTX Q1 Stage 16 — Preflight, Repository Freeze, and Integrity Gate

Run once before the parallel experiment lanes. It freezes the repository at the commit inspected while this notebook suite was built and verifies the current gate/broker implementation.

**Evidence rule:** failed/incomplete third-party infrastructure is recorded as such; no missing score is converted into a model failure or a success.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib, zipfile, random
BUNDLE = Path(os.environ.get("NTX_NOTEBOOK_BUNDLE", ".")).resolve()
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))
from ntx_q1_shared import *
ROOT = ensure_dir(DEFAULT_ROOT)
RESULTS_ROOT = ensure_dir(DEFAULT_RESULTS)
SEED = int(os.environ.get("NTX_SEED", "42"))
random.seed(SEED)
print("ROOT:", ROOT)
print("RESULTS:", RESULTS_ROOT)
print("Frozen repo commit:", REPO_COMMIT)

In [ ]:
STAGE = "16_preflight"
OUT = ensure_dir(RESULTS_ROOT / STAGE)
REPO = ROOT / "repo"
SELFTEST = os.environ.get("NTX_SELFTEST","0") == "1"

if SELFTEST:
    print("SELFTEST mode: network clone/install skipped.")
    write_json(OUT/"environment.json", env_report())
else:
    git_clone_frozen(REPO)
    print(run(["git","status","--short"], cwd=REPO).stdout)
    # Install exact repository package and analysis dependencies.
    run([sys.executable,"-m","pip","install","-q","-e",str(REPO/"niyamtrace")+"[dev,data,postgres]"])
    run([sys.executable,"-m","pip","install","-q","scipy","statsmodels","krippendorff","papermill","psutil"])
    write_json(OUT/"environment.json", env_report())

In [ ]:
checks = []
def add(name, ok, detail=""):
    checks.append({"check":name, "ok":bool(ok), "detail":str(detail)})

if SELFTEST:
    add("shared_module", True, "offline selftest")
else:
    sys.path.insert(0, str(REPO/"niyamtrace"))
    try:
        from packages.gate.gate import NiyamGate
        add("import_NiyamGate", True)
    except Exception as e:
        add("import_NiyamGate", False, repr(e))
    try:
        from packages.gate.broker import VerifiedExecutionBroker, BrokerVerificationError
        add("import_VerifiedExecutionBroker", True)
    except Exception as e:
        add("import_VerifiedExecutionBroker", False, repr(e))
    try:
        from apps.gateway.pipeline import NiyamPipeline, PipelineRequest
        add("import_NiyamPipeline", True)
    except Exception as e:
        add("import_NiyamPipeline", False, repr(e))

    broker_test = REPO/"niyamtrace/tests/security/test_e2e_broker.py"
    if broker_test.exists():
        cp = run([sys.executable,"-m","pytest",str(broker_test),"-q"], cwd=REPO/"niyamtrace", check=False, timeout=180)
        (OUT/"broker_pytest.log").write_text(cp.stdout, encoding="utf-8")
        add("broker_pytest", cp.returncode == 0, cp.stdout[-2000:])
    else:
        add("broker_pytest", False, "test_e2e_broker.py not found")

import pandas as pd
df = pd.DataFrame(checks)
df.to_csv(OUT/"preflight_checks.csv", index=False)
display(df)
hard_fail = [r for r in checks if r["check"].startswith("import_") and not r["ok"]]
record_status(OUT, STAGE, "PASS" if not hard_fail else "FAIL", hard_fail=hard_fail, frozen_commit=REPO_COMMIT)

In [ ]:
z = package_stage(OUT, "NTX_Q1_16_PRECHECK_RESULTS.zip")
print("Created:", z)
maybe_colab_download(z)